# XLM-RoBERTa fine-tune — TNL6323 (Colab GPU)

Runs the canonical `src/finetune_xlmr.py` from the repo so there is **no divergent copy**.

**What it does:** fine-tunes `xlm-roberta-base` (3-class sentiment) on the 960-row
`data/labeled/labeled_main_train.csv` only. A validation slice is carved *from the train split*,
so `labeled_main_test.csv` stays unseen and the in-domain score from `src/evaluate.py` is fair
and leak-free (matches the LogReg baseline's protocol).

**Before you run:** Runtime → Change runtime type → **GPU** (T4 free tier is enough; ~5 epochs in a few minutes).

Output: `models/xlmr_final/`. Download it (or save to Drive), drop it into the repo's `models/`
locally, then run `python src/evaluate.py` for the 3-tier table.

In [1]:
# 0. Confirm a GPU is attached (else training is very slow).
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0))

GPU: NVIDIA L4


In [2]:
# 1. Get the repo (private) + training data. Paste a GitHub Personal Access Token when prompted.
#    The token is read via getpass (not echoed) and only used for this clone.
from getpass import getpass
import os
token = getpass('GitHub PAT (repo read scope): ').strip()
os.system(f'git clone https://{token}@github.com/FarisMD22/tnl6323-project.git')
%cd tnl6323-project
!ls data/labeled/labeled_main_train.csv && echo 'train data present'

# --- Alternative if you don't want to use a token ---
# Skip this cell, then manually upload finetune_xlmr.py (to src/) and
# labeled_main_train.csv (to data/labeled/) via the Files panel, e.g.:
#   from google.colab import files; files.upload()

/content/tnl6323-project
data/labeled/labeled_main_train.csv
train data present


In [3]:
# 2. Install the training deps (Colab already ships a GPU build of torch).
!pip -q install -U 'transformers>=4.40' datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 138.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.2 MB/s eta 0:00:00:00:0100:01


In [4]:
# 3. (Optional) 1-step sanity check that the pipeline runs before the real train.
!python src/finetune_xlmr.py --smoke

train rows: 9 | val rows: 3 (held-out test stays in labeled_main_test.csv, scored later by src/evaluate.py)
train balance: {'negative': 3, 'positive': 3, 'neutral': 3}
config.json: 100% 615/615 [00:00<00:00, 2.75MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 117kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:01<00:00, 4.29MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:01<00:00, 8.38MB/s]
Map: 100% 9/9 [00:00<00:00, 1342.99 examples/s]
Map: 100% 3/3 [00:00<00:00, 674.00 examples/s]
model.safetensors: 100% 1.12G/1.12G [00:06<00:00, 162MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 5665.18it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPEC

In [5]:
# 4. Real fine-tune (5 epochs, fp16 on GPU). Saves to models/xlmr_final/.
!python src/finetune_xlmr.py

train rows: 816 | val rows: 144 (held-out test stays in labeled_main_test.csv, scored later by src/evaluate.py)
train balance: {'positive': 272, 'negative': 272, 'neutral': 272}
Map: 100% 816/816 [00:00<00:00, 10528.63 examples/s]
Map: 100% 144/144 [00:00<00:00, 8389.66 examples/s]
Loading weights: 100% 197/197 [00:00<00:00, 6753.12it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

N

In [6]:
  !ls -lh models/xlmr_final

total 1.1G
-rw-r--r-- 1 root root  970 Jun 24 06:23 config.json
-rw------- 1 root root 1.1G Jun 24 06:23 model.safetensors
-rw-r--r-- 1 root root  343 Jun 24 06:23 tokenizer_config.json
-rw-r--r-- 1 root root  17M Jun 24 06:23 tokenizer.json
-rw-r--r-- 1 root root 5.1K Jun 24 06:23 training_args.bin


In [7]:
# 5a. Save the model to Google Drive (recommended — the fine-tuned model is ~1.1 GB).
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/tnl6323/models' && cp -r models/xlmr_final '/content/drive/MyDrive/tnl6323/models/'
print('Saved to Drive: MyDrive/tnl6323/models/xlmr_final')

Mounted at /content/drive
Saved to Drive: MyDrive/tnl6323/models/xlmr_final


In [8]:
# 5b. Alternative: zip + browser download (slower for ~1 GB).
# !zip -r -q xlmr_final.zip models/xlmr_final
# from google.colab import files; files.download('xlmr_final.zip')

## Back on your machine
1. Put the downloaded folder at `models/xlmr_final/` in the repo (it's gitignored — don't commit the 1 GB model).
2. `python src/evaluate.py` — the 3-tier table now has an **XLM-R** row alongside LogReg.
3. `python src/aspect_degradation.py` — emits the XLM-R aspect heatmap too.
4. Update `REPORT_NOTES.md` §6 / `PROGRESS.md` with the XLM-R numbers, and re-check whether the
   aspect-conditioned flagship claim is supportable (still likely hypothesis-only given thin
   per-aspect support — see `REPORT_NOTES.md` §5).